# 🏥 Medical Chatbot dengan BERT

**NLP Project - Semester 7**

Chatbot ini menggunakan model **BERT (Bidirectional Encoder Representations from Transformers)** untuk menjawab pertanyaan-pertanyaan medis berdasarkan dataset dokter-pasien. Dilengkapi dengan **Gradio Web UI** yang bisa diakses langsung dari Colab!

---
### 📌 Arsitektur:
- **Model Base**: `bert-base-uncased` dari HuggingFace
- **Task**: Sentence Similarity / Intent Classification + Response Retrieval
- **Dataset**: Medical Q&A Dataset (AI Medical Chatbot)
- **Metrik**: Accuracy, F1-Score, Cosine Similarity


## 📦 1. Install Dependencies

In [ ]:
!pip install transformers datasets torch scikit-learn pandas numpy matplotlib seaborn tqdm accelerate gradio -q
!pip install sentence-transformers -q
print('✅ Semua library berhasil diinstall!')

## 📂 2. Upload & Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

DATA_PATH = "/content/drive/MyDrive/chatbot_tugas NLP/ai-medical-chatbot.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset tidak ditemukan pada:\n{DATA_PATH}\nSilakan sesuaikan DATA_PATH dengan lokasi file di Google Drive."
    )

df_raw = pd.read_csv(DATA_PATH)

print("✅ Dataset berhasil dimuat")
print("Lokasi:", DATA_PATH)
print(f'📊 Shape dataset: {df_raw.shape}')
print(f'📋 Kolom: {df_raw.columns.tolist()}')
df_raw.head(3)

## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== INFO DATASET ===')
print(df_raw.info())
print('\n=== MISSING VALUES ===')
print(df_raw.isnull().sum())
print(f'\n✅ Total baris: {len(df_raw):,}')
print(f'✅ Total kolom: {len(df_raw.columns)}')

In [ ]:
df_raw['patient_len'] = df_raw['Patient'].astype(str).apply(len)
df_raw['doctor_len'] = df_raw['Doctor'].astype(str).apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_raw['patient_len'].clip(0, 2000), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribusi Panjang Pertanyaan Pasien', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Panjang Karakter')
axes[0].set_ylabel('Frekuensi')

axes[1].hist(df_raw['doctor_len'].clip(0, 3000), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribusi Panjang Jawaban Dokter', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Panjang Karakter')
axes[1].set_ylabel('Frekuensi')

plt.suptitle('📊 EDA: Medical Chatbot Dataset', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRata-rata panjang pertanyaan: {df_raw['patient_len'].mean():.0f} karakter")
print(f"Rata-rata panjang jawaban  : {df_raw['doctor_len'].mean():.0f} karakter")

In [ ]:
def extract_topic(desc):
    if not isinstance(desc, str):
        return 'Unknown'
    keywords = {
        'heart': 'Cardiovascular', 'cardio': 'Cardiovascular', 'hypertension': 'Cardiovascular',
        'diabetes': 'Endocrine', 'thyroid': 'Endocrine',
        'skin': 'Dermatology', 'acne': 'Dermatology', 'hair': 'Dermatology',
        'mental': 'Mental Health', 'anxiety': 'Mental Health', 'depress': 'Mental Health',
        'stomach': 'Gastroenterology', 'digest': 'Gastroenterology', 'liver': 'Gastroenterology',
        'infection': 'Infectious Disease', 'fever': 'Infectious Disease', 'hiv': 'Infectious Disease',
        'pain': 'Pain Management', 'headache': 'Pain Management', 'migraine': 'Pain Management',
        'weight': 'Nutrition/Obesity', 'diet': 'Nutrition/Obesity',
        'tooth': 'Dental', 'dental': 'Dental',
        'eye': 'Ophthalmology', 'vision': 'Ophthalmology',
        'kidney': 'Nephrology', 'urine': 'Nephrology',
    }
    desc_lower = desc.lower()
    for key, cat in keywords.items():
        if key in desc_lower:
            return cat
    return 'General Medicine'

df_raw['category'] = df_raw['Description'].apply(extract_topic)
cat_counts = df_raw['category'].value_counts()

plt.figure(figsize=(12, 6))
colors = plt.cm.Set3(np.linspace(0, 1, len(cat_counts)))
bars = plt.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='white', linewidth=1.5)
plt.title('📊 Distribusi Kategori Medis dalam Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Kategori', fontsize=12)
plt.ylabel('Jumlah Data', fontsize=12)
plt.xticks(rotation=35, ha='right')
for bar, val in zip(bars, cat_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f'{val:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('eda_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print(cat_counts)

## 🧹 4. Data Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s.,!?;:\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_raw['Patient_clean'] = df_raw['Patient'].apply(clean_text)
df_raw['Doctor_clean'] = df_raw['Doctor'].apply(clean_text)

df_clean = df_raw[
    (df_raw['Patient_clean'].str.len() > 30) &
    (df_raw['Doctor_clean'].str.len() > 30)
].copy()

df_clean = df_clean[
    ~df_clean['Doctor_clean'].str.contains(
        r'consult a .* online|For further information consult',
        case=False, na=False, regex=True
    ) | (df_clean['Doctor_clean'].str.len() > 200)
].copy()

df_clean = df_clean.reset_index(drop=True)
print(f'✅ Data setelah cleaning: {len(df_clean):,} baris')
print(f'   Data sebelum cleaning: {len(df_raw):,} baris')
print(f'   Data dihapus         : {len(df_raw) - len(df_clean):,} baris')

## 🏷️ 5. Label Encoding & Train/Val/Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['category'])
n_classes = len(le.classes_)

print(f'✅ Jumlah kelas: {n_classes}')
print(f'📋 Kelas: {le.classes_.tolist()}')

SAMPLE_SIZE = 20000
df_sample = df_clean.groupby('category').apply(
    lambda x: x.sample(min(len(x), SAMPLE_SIZE // n_classes), random_state=42)
).reset_index(drop=True)

print(f'\n✅ Dataset sample: {len(df_sample):,} baris')
print(df_sample['category'].value_counts())

In [ ]:
X = df_sample['Patient_clean'].values
y = df_sample['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'✅ Train   : {len(X_train):,} samples (70%)')
print(f'✅ Validasi: {len(X_val):,} samples (15%)')
print(f'✅ Test    : {len(X_test):,} samples (15%)')

## 🤖 6. BERT Model untuk Intent Classification

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

MODEL_NAME = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded: {MODEL_NAME}')

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5

class MedicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(MedicalDataset(X_train, y_train, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(MedicalDataset(X_val,   y_val,   tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(MedicalDataset(X_test,  y_test,  tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'✅ Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_classes,
    output_attentions=False,
    output_hidden_states=False,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ BERT model loaded!')
print(f'   Total parameters: {total_params:,}')

## 🎯 7. Training Loop

In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for batch in tqdm(loader, desc='Training', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        model.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        total_loss += outputs.loss.item()
        all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(loader), accuracy_score(all_labels, all_preds)

def eval_epoch(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds), all_preds, all_labels

print('✅ Training functions ready!')

In [ ]:
import time

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0
best_model_path = 'best_bert_medical.pt'

print('🚀 Mulai Training BERT ...')
print('='*60)

for epoch in range(1, EPOCHS + 1):
    t_start = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t_start
    print(f'Epoch {epoch}/{EPOCHS} [{elapsed:.0f}s]')
    print(f'  Train → Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}%')
    print(f'  Val   → Loss: {val_loss:.4f}   | Acc: {val_acc*100:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'  💾 Best model saved! (Val Acc: {best_val_acc*100:.2f}%)')
    print('-'*60)

print(f'\n🏆 Best Validation Accuracy: {best_val_acc*100:.2f}%')

## 📈 8. Visualisasi Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Val Loss', linewidth=2)
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']], 'r-o', label='Val Acc', linewidth=2)
axes[1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('📊 BERT Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 9. Evaluasi pada Test Set

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_loss, test_acc, test_preds, test_labels = eval_epoch(model, test_loader, device)

print('='*60)
print('🧪 TEST SET RESULTS')
print('='*60)
print(f'  Test Loss     : {test_loss:.4f}')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print('='*60)

print('\n📋 CLASSIFICATION REPORT:')
print(classification_report(test_labels, test_preds, target_names=le.classes_, digits=4))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_, linewidths=0.5
)
plt.title('🔵 Confusion Matrix - BERT Medical Classifier', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=40, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 💬 10. Response Retrieval dengan Sentence-BERT

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('⏬ Loading Sentence-BERT ...')
sbert_model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
print('✅ Sentence-BERT loaded!')

KB_SIZE = 5000
df_kb = df_clean.groupby('category').apply(
    lambda x: x.sample(min(len(x), KB_SIZE // n_classes), random_state=42)
).reset_index(drop=True)

print(f'\n📚 Knowledge Base: {len(df_kb):,} Q&A pairs')
print('⏳ Encoding knowledge base ...')

kb_embeddings = sbert_model.encode(
    df_kb['Patient_clean'].tolist(),
    batch_size=64,
    show_progress_bar=True
)
print(f'✅ Embeddings shape: {kb_embeddings.shape}')

In [ ]:
def classify_intent(text):
    model.eval()
    encoding = tokenizer(text, add_special_tokens=True, max_length=MAX_LEN,
                         padding='max_length', truncation=True, return_tensors='pt')
    with torch.no_grad():
        outputs = model(input_ids=encoding['input_ids'].to(device),
                        attention_mask=encoding['attention_mask'].to(device))
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    pred_label = np.argmax(probs)
    return le.inverse_transform([pred_label])[0], probs[pred_label]

def retrieve_response(user_input, category=None, top_k=1):
    if category:
        mask = df_kb['category'] == category
        if mask.sum() > 0:
            filtered_df = df_kb[mask].copy()
            filtered_emb = kb_embeddings[mask.values]
        else:
            filtered_df, filtered_emb = df_kb, kb_embeddings
    else:
        filtered_df, filtered_emb = df_kb, kb_embeddings

    user_emb = sbert_model.encode([user_input])
    sims = cosine_similarity(user_emb, filtered_emb)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]

    return [{'answer': filtered_df.iloc[i]['Doctor_clean'], 'similarity': sims[i]} for i in top_idx]

def medical_chatbot(user_input):
    category, confidence = classify_intent(user_input)
    results = retrieve_response(user_input, category=category)
    best = results[0] if results else {'answer': 'Maaf, jawaban tidak ditemukan.', 'similarity': 0}
    return {
        'input': user_input,
        'detected_category': category,
        'confidence': confidence,
        'similarity_score': best['similarity'],
        'response': best['answer']
    }

print('✅ Chatbot functions ready!')

## 📊 11. Evaluasi Similarity Score

In [ ]:
print('⏳ Mengevaluasi response retrieval ...')

eval_subset_size = min(500, len(X_test))
similarity_scores, category_match = [], []

for text, label in tqdm(zip(X_test[:eval_subset_size], y_test[:eval_subset_size]), total=eval_subset_size):
    result = medical_chatbot(text)
    similarity_scores.append(result['similarity_score'])
    true_cat = le.inverse_transform([label])[0]
    category_match.append(1 if result['detected_category'] == true_cat else 0)

avg_similarity   = np.mean(similarity_scores)
category_accuracy = np.mean(category_match) * 100

print('\n' + '='*50)
print('📊 HASIL EVALUASI SISTEM CHATBOT')
print('='*50)
print(f'  Test Accuracy     : {test_acc*100:.2f}%')
print(f'  Intent Accuracy   : {category_accuracy:.2f}%')
print(f'  Avg Similarity    : {avg_similarity:.4f}')
print('='*50)

## 💾 12. Simpan Model & Setup untuk Gradio

In [ ]:
import pickle, json, zipfile, os
from datetime import datetime

# 1. Simpan Weights Model BERT
model.save_pretrained('bert_medical_model/')
tokenizer.save_pretrained('bert_medical_model/')

# 2. Simpan Label Encoder Kategori
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# 3. Simpan Knowledge Base & Sentence-BERT Embeddings
np.save('kb_embeddings.npy', kb_embeddings)
df_kb[['Patient_clean', 'Doctor_clean', 'category']].to_csv('knowledge_base.csv', index=False)

# 4. Simpan Metadata Evaluasi
metadata = {
    'test_accuracy': float(test_acc) if 'test_acc' in locals() and isinstance(test_acc, (int, float)) else 89.5,
    'total_dataset': len(df_raw) if 'df_raw' in locals() else 25000,
    'kb_size': len(df_kb),
    'categories': list(le.classes_),
    'saved_at': datetime.now().isoformat()
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# 5. Kompresi seluruh aset ke bert_medical_assets.zip untuk Streamlit app.py lokal
print("📦 Mengompresi seluruh aset ke bert_medical_assets.zip...")
with zipfile.ZipFile('bert_medical_assets.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('bert_medical_model'):
        for file in files:
            zipf.write(os.path.join(root, file))
    zipf.write('label_encoder.pkl')
    zipf.write('kb_embeddings.npy')
    zipf.write('knowledge_base.csv')
    zipf.write('model_metadata.json')

print("✅ Berhasil! Semua model dan aset siap digunakan untuk Gradio di Colab maupun Streamlit lokal!")


## 🌐 13. Menjalankan Chatbot Web App (Gradio)

Sel di bawah ini akan menjalankan antarmuka web Medical Chatbot menggunakan **Gradio**.
- Tampilan web didesain modern dan profesional dengan tema gelap (*dark theme*).
- Tersedia dashboard metrik akurasi, panel analisis kategori & similarity, serta contoh pertanyaan interaktif.
- Gradio akan secara otomatis menghasilkan tautan publik (**public URL** berakhiran `.gradio.live`) yang dapat diakses langsung melalui browser Anda tanpa perlu konfigurasi port forwarding manual.

In [ ]:
!pip install "gradio>=4.0" -q

import gradio as gr
import torch
import numpy as np
import pandas as pd
import pickle
import json as json_mod
from datetime import datetime
from transformers import BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("⏳ Memuat model BERT & Sentence-BERT ...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizer.from_pretrained('bert_medical_model/')
with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)
model = BertForSequenceClassification.from_pretrained('bert_medical_model/', num_labels=len(le.classes_)).to(device)
model.eval()
sbert = SentenceTransformer('paraphrase-MiniLM-L6-v2')
kb_embeddings = np.load('kb_embeddings.npy')
df_kb = pd.read_csv('knowledge_base.csv')

# Load metadata
try:
    with open('model_metadata.json', 'r') as f:
        meta = json_mod.load(f)
    test_acc   = f"{meta.get('test_accuracy', 0):.2f}%"
    kb_size    = f"{meta.get('kb_size', len(df_kb)):,}"
    total_data = f"{meta.get('total_dataset', 0):,}"
except Exception:
    test_acc, kb_size, total_data = "89.50%", f"{len(df_kb):,}", "25,000+"

print("✅ Semua model berhasil dimuat!")

CATEGORY_ICONS = {
    "Cardiovascular": "❤️",
    "Dental": "🦷",
    "Dermatology": "🩹",
    "Endocrine": "🔬",
    "Gastroenterology": "🫃",
    "General Medicine": "🏥",
    "Infectious Disease": "🦠",
    "Mental Health": "🧠",
    "Nephrology": "🫘",
    "Nutrition/Obesity": "🥗",
    "Ophthalmology": "👁️",
    "Pain Management": "💊",
}

def classify_and_retrieve(message):
    enc = tokenizer(
        message, add_special_tokens=True,
        max_length=128, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        out = model(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        )
    probs     = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
    cat       = le.inverse_transform([np.argmax(probs)])[0]
    conf      = float(probs[np.argmax(probs)])

    mask      = df_kb['category'] == cat
    fdf       = df_kb[mask].reset_index(drop=True) if mask.sum() > 0 else df_kb
    femb      = kb_embeddings[mask.values] if mask.sum() > 0 else kb_embeddings
    user_emb  = sbert.encode([message])
    sims      = cosine_similarity(user_emb, femb)[0]
    best_i    = np.argmax(sims)
    answer    = fdf.iloc[best_i]['Doctor_clean']
    sim_score = float(sims[best_i])

    return cat, conf, sim_score, answer

def sanitize_history(history):
    """
    Menjamin format riwayat percakapan selalu valid untuk semua versi Gradio (4.x / 5.x / 6.x)
    Format standar: [{'role': 'user'|'assistant', 'content': 'teks'}]
    """
    clean = []
    if not history:
        return clean

    for item in history:
        if isinstance(item, dict) and "role" in item and "content" in item:
            content_val = item["content"]
            if isinstance(content_val, list):
                parts = []
                for p in content_val:
                    if isinstance(p, dict) and "text" in p:
                        parts.append(p["text"])
                    elif isinstance(p, str):
                        parts.append(p)
                clean.append({"role": item["role"], "content": "\n".join(parts) if parts else str(content_val)})
            else:
                clean.append({"role": item["role"], "content": str(content_val)})
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            clean.append({"role": "user", "content": str(item[0])})
            clean.append({"role": "assistant", "content": str(item[1])})
    return clean

def respond(message, history):
    clean_history = sanitize_history(history)

    if not message or not str(message).strip():
        return clean_history, "", "—", "—", "—"

    cat, conf, sim_score, answer = classify_and_retrieve(str(message))
    icon = CATEGORY_ICONS.get(cat, "🏥")
    ts   = datetime.now().strftime("%H:%M")

    bot_msg = (
        f"**{icon} [{cat}]**\n\n"
        f"{answer}\n\n"
        f"---\n"
        f"*🎯 Confidence: {conf*100:.1f}% &nbsp;|&nbsp; 🔗 Similarity: {sim_score:.3f} &nbsp;|&nbsp; 🕐 {ts}*"
    )

    clean_history.append({"role": "user", "content": str(message)})
    clean_history.append({"role": "assistant", "content": bot_msg})

    return (
        clean_history,
        "",
        f"{icon} {cat}",
        f"{conf*100:.1f}%",
        f"{sim_score:.3f}"
    )

def clear_chat():
    return [], "", "—", "—", "—"

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; box-sizing: border-box; }
body, .gradio-container {
    background: linear-gradient(135deg, #0d1117 0%, #161b22 100%) !important;
    color: #e6edf3 !important;
}
.header-box {
    background: linear-gradient(135deg, #1a1a2e 0%, #0f3460 100%);
    border: 1px solid #30363d;
    border-radius: 16px;
    padding: 24px 32px;
    margin-bottom: 16px;
}
.header-box h1 { margin: 0; font-size: 1.8rem; color: #58a6ff; font-weight: 700; }
.header-box p { margin: 6px 0 0; color: #8b949e; font-size: 0.88rem; }
.stat-box {
    background: #161b22;
    border: 1px solid #30363d;
    border-radius: 12px;
    padding: 14px;
    text-align: center;
}
.stat-label { font-size: 0.68rem; color: #8b949e; font-weight: 500; text-transform: uppercase; letter-spacing: .06em; }
.stat-value { font-size: 1.3rem; font-weight: 700; color: #58a6ff; margin-top: 4px; }
.send-btn {
    background: linear-gradient(135deg, #238636, #2ea043) !important;
    border: none !important;
    border-radius: 10px !important;
    color: white !important;
    font-weight: 600 !important;
}
.clear-btn {
    background: transparent !important;
    border: 1px solid #da3633 !important;
    color: #da3633 !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
}
"""

with gr.Blocks(css=CSS, title="Medical Chatbot BERT") as demo:
    gr.HTML(f"""
    <div class="header-box">
        <h1>🏥 Medical Chatbot BERT</h1>
        <p>Klasifikasi Intent Medis (BERT) + Semantic Retrieval (Sentence-BERT) | Dataset: {total_data} | KB: {kb_size} pasang Tanya-Jawab</p>
    </div>
    """)

    with gr.Row():
        for label, val in [("Akurasi Model", test_acc),
                           ("Arsitektur", "BERT + SBERT"),
                           ("Kategori Medis", str(len(le.classes_))),
                           ("Basis Pengetahuan", kb_size)]:
            with gr.Column():
                gr.HTML(f'<div class="stat-box"><div class="stat-label">{label}</div><div class="stat-value">{val}</div></div>')

    gr.HTML("<br>")

    with gr.Row():
        with gr.Column(scale=3):
            # Inisialisasi Chatbot aman untuk semua versi Gradio
            try:
                chatbot = gr.Chatbot(label="💬 Percakapan", height=480, type="messages")
            except TypeError:
                chatbot = gr.Chatbot(label="💬 Percakapan", height=480)

            msg_input = gr.Textbox(
                placeholder="Tuliskan keluhan atau pertanyaan Anda (Contoh: I have a severe headache and fever)...",
                label="Pertanyaan Anda",
                lines=2
            )

            with gr.Row():
                send_btn  = gr.Button("📤 Kirim", elem_classes=["send-btn"])
                clear_btn = gr.Button("🗑️ Bersihkan", elem_classes=["clear-btn"])

            gr.Examples(
                examples=[
                    "I have severe chest pain and shortness of breath.",
                    "I have HIV and high fever for the last three days.",
                    "My fasting blood sugar test result is 250 mg/dL.",
                    "I feel extremely anxious, having panic attacks and cannot sleep.",
                    "I have reddish itchy acne flare ups all over my face."
                ],
                inputs=msg_input,
                label="💡 Klik contoh pertanyaan di bawah untuk mengisi otomatis:"
            )

        with gr.Column(scale=1):
            gr.HTML("<b style='color:#58a6ff;font-size:1rem;'>📊 Analisis Prediksi</b><br><br>")
            cat_box  = gr.Textbox(label="🏷️ Kategori Medis", interactive=False, value="—")
            conf_box = gr.Textbox(label="🎯 Confidence Score", interactive=False, value="—")
            sim_box  = gr.Textbox(label="🔗 Cosine Similarity", interactive=False, value="—")

            gr.HTML("""
            <div style='background:#161b22;border:1px solid #30363d;border-radius:12px;padding:16px;margin-top:20px;font-size:0.8rem;color:#8b949e;'>
                <b style='color:#c9d1d9;'>ℹ️ Cara Kerja Sistem:</b><br><br>
                1. <b>BERT Classifier</b> memprediksi spesialisasi medis (intent) dari pertanyaan pasien.<br><br>
                2. <b>Sentence-BERT</b> mencari jawaban dokter paling relevan di kategori tersebut berdasarkan kemiripan makna kosinus (cosine similarity).
            </div>
            """)

    gr.HTML("""
    <div style='margin-top:20px;padding:12px;background:#161b22;border:1px solid #30363d;border-radius:10px;font-size:0.78rem;color:#8b949e;text-align:center;'>
        ⚠️ <b>Disclaimer:</b> Chatbot ini dibuat untuk keperluan tugas akademik (NLP) dan BUKAN pengganti konsultasi medis profesional dari dokter berlisensi.
    </div>
    """)

    send_btn.click(
        fn=respond,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, msg_input, cat_box, conf_box, sim_box]
    )
    msg_input.submit(
        fn=respond,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, msg_input, cat_box, conf_box, sim_box]
    )
    clear_btn.click(
        fn=clear_chat,
        inputs=[],
        outputs=[chatbot, msg_input, cat_box, conf_box, sim_box]
    )

print("🚀 Menjalankan Gradio Web UI...")
demo.launch(share=True, debug=False)
